In [4]:
# Import section
from backtesting import Backtest
import pandas as pd
from backtesting import Strategy
from backtesting.lib import crossover
import numpy as np
import math

# META
MARGIN = 0.01
CASH = 10_000
COMMISSION = 0.00002
table_name = 'f0_atr_sl'

# DEFINE
trade_histories = pd.DataFrame({
  "datetime": [],
  "result": []
})

# Strategy

In [5]:
# Define the EMA Strategy
class STRAT(Strategy):
    n1 = 9  # Fast EMA period
    n2 = 21  # Slow EMA period

    risk_pct = 0.01      # Risk 1% of account per trade
    risk_reward = 1      # TP = x the SL distance
    atr_period = 14

    def init(self):
        super().init()

        # Calculate EMAs using Pandas directly
        close = pd.Series(self.data.Close)
        high = pd.Series(self.data.High)
        low = pd.Series(self.data.Low)

        self.ema1 = self.I(lambda: close.ewm(span=self.n1, adjust=False).mean(), name="ema fast")
        self.ema2 = self.I(lambda: close.ewm(span=self.n2, adjust=False).mean(), name="ema slow")

        if self.atr_period > 0:
            prev_close = close.shift(1)
            tr = pd.concat([
                high - low,
                (high - prev_close).abs(),
                (low - prev_close).abs()
            ], axis=1).max(axis=1)
            self.atr = self.I(lambda: tr.ewm(span=self.atr_period, adjust=False).mean(), name="atr")
        else:
            self.atr = self.I(lambda: pd.Series(0.0, index=close.index), name="atr")

    def next(self):
        price = self.data.Close[-1]

        prev_low  = self.data.Low[-2]
        prev_high = self.data.High[-2]

        long_signal  = crossover(self.ema1, self.ema2) 
        short_signal = crossover(self.ema2, self.ema1)

        risk_amt = max(self.equity, CASH) * self.risk_pct

        def get_size_by_leveraged(sl_dist):
            ideal_size = risk_amt / sl_dist
            # We subtract a buffer 85%
            max_leverage_size = (self.equity * 0.85) / (price * MARGIN)
            size = int(min(ideal_size, max_leverage_size))
            return size

        if long_signal:

            sl_dist = self.atr[-1]
            tp_dist = sl_dist * self.risk_reward

            if sl_dist == 0:
                return

            size = get_size_by_leveraged(sl_dist)

            if size <= 0:
                return

            self.position.close()
            self.buy(size=size, sl=price - sl_dist, tp=price + tp_dist)

        elif short_signal:

            sl_dist = self.atr[-1]
            tp_dist = sl_dist * self.risk_reward

            if sl_dist == 0:
                return

            size = get_size_by_leveraged(sl_dist)

            if size <= 0:
                return


            self.position.close()
            self.sell(size=size, sl=price + sl_dist, tp=price - tp_dist)


# Basic backtest

In [6]:
# Load sample data and run a basic backtest
file_path = "~/market_data/XAUUSD/XAUUSD_M5.csv"

try:
  # Load data
  data = pd.read_csv(file_path, parse_dates=['Datetime'], index_col='Datetime')
  
  # Run backtest
  bt = Backtest(data, STRAT, cash=CASH, commission=COMMISSION, margin=MARGIN)
  stats = bt.run()
  
  print(stats)
  
except Exception as e:
  print(f"Error: {str(e)}")

Start                     2020-01-02 01:00:00
End                       2025-12-31 23:55:00
Duration                   2190 days 22:55:00
Exposure Time [%]                     0.92633
Equity Final [$]                     18.12479
Equity Peak [$]                   10554.71447
Commissions [$]                    5199.66565
Return [%]                          -99.81875
Buy & Hold Return [%]               184.03997
Return (Ann.) [%]                   -64.61387
Volatility (Ann.) [%]                42.87768
CAGR [%]                            -65.09145
Sharpe Ratio                         -1.50693
Sortino Ratio                        -0.97378
Calmar Ratio                         -0.64725
Alpha [%]                           -95.67539
Beta                                 -0.02251
Max. Drawdown [%]                   -99.82828
Avg. Drawdown [%]                    -7.49916
Max. Drawdown Duration     2177 days 05:55:00
Avg. Drawdown Duration      128 days 20:04:00
# Trades                          

# SQLite connection

In [7]:
import sqlite3

# Setup SQLite database
db_path = "../backtest_results.db"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

print(f"Database setup complete at {db_path}")

conn.commit()

Database setup complete at ../backtest_results.db


# Combination test

In [ ]:
from itertools import product
from collections import namedtuple


cursor.execute(f'''
CREATE TABLE IF NOT EXISTS {table_name} (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    asset TEXT NOT NULL,
    timeframe TEXT NOT NULL,
    n1 INTEGER NOT NULL,
    n2 INTEGER NOT NULL,
    risk_pct REAL NOT NULL,
    risk_reward REAL NOT NULL,
    atr_period INTEGER NOT NULL,
    return_pct REAL,
    max_drawdown_pct REAL,
    cagr_pct REAL,
    sharpe_ratio REAL,
    profit_factor REAL,
    win_rate_pct REAL,
    num_trades INTEGER,
    equity_final REAL,
    run_timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
)
''')

# Test all asset/timeframe combinations
assets = ['XAUUSD', 'EURUSD', 'GBPUSD', 'USDCAD', 'AUDUSD', 'USDCHF', 'US100', 'US500']
timeframes = ['M1', 'M5', 'M15', 'H1', 'H4', 'D1']

# Strategy parameters (from current STRAT class)
init_params = {
    'n1': 9,
    'n2': 21,
    'risk_pct': 0.01,
    'risk_reward': 0.5,
    'atr_period': 1
}

parameters = [
    timeframes,
    assets,
    [0.25, 0.33, 0.5, *range(1, 4)],  # rr
    [0.005, 0.01]  # risk percent
]

total_combinations = math.prod(len(p) for p in parameters)
print(f"Testing {total_combinations} combinations...")

for timeframe, asset, rr, risk_pct in product(*parameters):
    file_path = f"~/market_data/{asset}/{asset}_{timeframe}.csv"
    
    try:
        # Load data
        data = pd.read_csv(file_path, parse_dates=['Datetime'], index_col='Datetime')
        
        # Run backtest
        bt = Backtest(data, STRAT, cash=CASH, commission=COMMISSION, margin=MARGIN)
        stats = bt.run(
            risk_reward=rr,
            risk_pct=risk_pct,
            atr_period=init_params['atr_period']
        )
        
        # Extract metrics
        return_pct = stats['Return [%]']
        max_drawdown_pct = stats['Max. Drawdown [%]']
        cagr_pct = stats['CAGR [%]']
        sharpe_ratio = stats['Sharpe Ratio']
        profit_factor = stats['Profit Factor']
        win_rate_pct = stats['Win Rate [%]']
        num_trades = stats['# Trades']
        equity_final = stats['Equity Final [$]']
        
        # Store in database
        cursor.execute(f'''
            INSERT INTO {table_name}
            (asset, timeframe, n1, n2, risk_pct, risk_reward, atr_period, return_pct, max_drawdown_pct, cagr_pct, sharpe_ratio, profit_factor, win_rate_pct, num_trades, equity_final)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', (
            asset,
            timeframe,
            init_params['n1'],
            init_params['n2'],
            risk_pct,
            rr,
            init_params['atr_period'],
            return_pct,
            max_drawdown_pct,
            cagr_pct,
            sharpe_ratio,
            profit_factor,
            win_rate_pct,
            num_trades,
            equity_final
        ))
        conn.commit()
        
        print(f"✓ {asset}_{timeframe}: Return={return_pct:.2f}%, DD={max_drawdown_pct:.2f}%, Sharpe={sharpe_ratio:.2f}, WinRate={win_rate_pct:.2f}%")
        
    except Exception as e:
        print(f"✗ {asset}_{timeframe}: Error - {str(e)}")
        # Store error record
        cursor.execute(f'''
            INSERT INTO {table_name}
            (asset, timeframe, n1, n2, risk_pct, risk_reward, atr_period, return_pct, max_drawdown_pct, cagr_pct, sharpe_ratio, profit_factor, win_rate_pct, num_trades, equity_final)
            VALUES (?, ?, ?, ?, ?, ?, ?, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL)
        ''', (
            asset,
            timeframe,
            init_params['n1'],
            init_params['n2'],
            risk_pct,
            rr,
            init_params['atr_period']
        ))
        conn.commit()

print("\nBacktesting complete!")
print(f"Results stored in {db_path}")

Testing 384 combinations...
✓ XAUUSD_M1: Return=-99.87%, DD=-99.87%, Sharpe=-3.53, WinRate=46.95%
✓ XAUUSD_M1: Return=-99.85%, DD=-99.85%, Sharpe=-3.41, WinRate=46.75%
✓ XAUUSD_M1: Return=-99.86%, DD=-99.86%, Sharpe=-3.42, WinRate=43.65%
✓ XAUUSD_M1: Return=-99.86%, DD=-99.86%, Sharpe=-3.25, WinRate=43.73%
✓ XAUUSD_M1: Return=-99.83%, DD=-99.84%, Sharpe=-2.37, WinRate=33.87%
✓ XAUUSD_M1: Return=-99.84%, DD=-99.86%, Sharpe=-3.16, WinRate=33.80%
✓ XAUUSD_M1: Return=-99.84%, DD=-99.86%, Sharpe=-1.43, WinRate=27.86%
✓ XAUUSD_M1: Return=-99.85%, DD=-99.89%, Sharpe=-2.07, WinRate=27.12%
✓ EURUSD_M1: Return=-100.00%, DD=-100.00%, Sharpe=-9.85, WinRate=40.74%
✓ EURUSD_M1: Return=-100.00%, DD=-100.00%, Sharpe=-9.67, WinRate=40.46%
✓ EURUSD_M1: Return=-100.00%, DD=-100.00%, Sharpe=-9.35, WinRate=42.06%
✓ EURUSD_M1: Return=-100.00%, DD=-100.00%, Sharpe=-9.44, WinRate=42.05%
✓ EURUSD_M1: Return=-100.00%, DD=-100.00%, Sharpe=-6.77, WinRate=31.09%
✓ EURUSD_M1: Return=-100.00%, DD=-100.00%, Sharpe=-6

/tmp/ipykernel_18866/1550430692.py:58: UserWarning: Some prices are larger than initial cash value. Note that fractional trading is not supported by this class. If you want to trade Bitcoin, increase initial cash, or trade μBTC or satoshis instead (see e.g. class `backtesting.lib.FractionalBacktest`.
  bt = Backtest(data, STRAT, cash=CASH, commission=COMMISSION, margin=MARGIN)


✓ US100_M1: Return=-98.85%, DD=-98.85%, Sharpe=-2.90, WinRate=56.18%


/tmp/ipykernel_18866/1550430692.py:58: UserWarning: Some prices are larger than initial cash value. Note that fractional trading is not supported by this class. If you want to trade Bitcoin, increase initial cash, or trade μBTC or satoshis instead (see e.g. class `backtesting.lib.FractionalBacktest`.
  bt = Backtest(data, STRAT, cash=CASH, commission=COMMISSION, margin=MARGIN)


✓ US100_M1: Return=-98.89%, DD=-98.89%, Sharpe=-2.51, WinRate=57.10%


...

✓ US500_H4: Return=63.90%, DD=-16.62%, Sharpe=0.64, WinRate=31.28%
✓ XAUUSD_D1: Return=-1.58%, DD=-3.01%, Sharpe=-0.30, WinRate=59.26%
✓ XAUUSD_D1: Return=-6.68%, DD=-7.46%, Sharpe=-0.55, WinRate=56.06%
✓ XAUUSD_D1: Return=-0.21%, DD=-2.12%, Sharpe=-0.03, WinRate=46.30%
✓ XAUUSD_D1: Return=-5.38%, DD=-8.13%, Sharpe=-0.33, WinRate=42.42%
✓ XAUUSD_D1: Return=2.28%, DD=-2.73%, Sharpe=0.23, WinRate=33.33%
✓ XAUUSD_D1: Return=-3.04%, DD=-8.80%, Sharpe=-0.14, WinRate=28.79%
✓ XAUUSD_D1: Return=0.65%, DD=-4.94%, Sharpe=0.06, WinRate=22.22%
✓ XAUUSD_D1: Return=-6.46%, DD=-14.37%, Sharpe=-0.25, WinRate=19.70%
✓ EURUSD_D1: Return=0.96%, DD=-1.76%, Sharpe=0.14, WinRate=68.42%
✓ EURUSD_D1: Return=1.91%, DD=-3.53%, Sharpe=0.14, WinRate=68.42%
✓ EURUSD_D1: Return=2.87%, DD=-3.49%, Sharpe=0.29, WinRate=54.39%
✓ EURUSD_D1: Return=5.75%, DD=-6.95%, Sharpe=0.29, WinRate=54.39%
✓ EURUSD_D1: Return=6.48%, DD=-6.18%, Sharpe=0.47, WinRate=40.35%
✓ EURUSD_D1: Return=13.17%, DD=-12.21%, Sharpe=0.46, WinRate=4

/tmp/ipykernel_18866/1550430692.py:59: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  stats = bt.run(
/tmp/ipykernel_18866/1550430692.py:59: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  stats = bt.run(
/tmp/ipykernel_18866/1550430692.py:59: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  stats = bt.run(
/tmp/ipykernel_18866/1550430692.py:59: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  stats = bt.run(
/tmp/ipykernel_18866/1550430692.py:58: UserWarning: Some prices are larger than initial cash value. Note that fractional trading is not supported by this class. If you want to trade Bitcoin, incre

✓ USDCHF_D1: Return=-9.45%, DD=-12.86%, Sharpe=-0.49, WinRate=20.45%
✓ USDCHF_D1: Return=-18.91%, DD=-25.64%, Sharpe=-0.48, WinRate=20.45%
✓ US100_D1: Return=0.00%, DD=-0.00%, Sharpe=nan, WinRate=nan%
✓ US100_D1: Return=-0.63%, DD=-0.63%, Sharpe=-0.44, WinRate=0.00%
✓ US100_D1: Return=0.00%, DD=-0.00%, Sharpe=nan, WinRate=nan%
✓ US100_D1: Return=-0.63%, DD=-0.63%, Sharpe=-0.44, WinRate=0.00%
✓ US100_D1: Return=0.00%, DD=-0.00%, Sharpe=nan, WinRate=nan%


/tmp/ipykernel_18866/1550430692.py:58: UserWarning: Some prices are larger than initial cash value. Note that fractional trading is not supported by this class. If you want to trade Bitcoin, increase initial cash, or trade μBTC or satoshis instead (see e.g. class `backtesting.lib.FractionalBacktest`.
  bt = Backtest(data, STRAT, cash=CASH, commission=COMMISSION, margin=MARGIN)
/tmp/ipykernel_18866/1550430692.py:58: UserWarning: Some prices are larger than initial cash value. Note that fractional trading is not supported by this class. If you want to trade Bitcoin, increase initial cash, or trade μBTC or satoshis instead (see e.g. class `backtesting.lib.FractionalBacktest`.
  bt = Backtest(data, STRAT, cash=CASH, commission=COMMISSION, margin=MARGIN)
/tmp/ipykernel_18866/1550430692.py:58: UserWarning: Some prices are larger than initial cash value. Note that fractional trading is not supported by this class. If you want to trade Bitcoin, increase initial cash, or trade μBTC or satoshis 

✓ US100_D1: Return=-0.63%, DD=-0.63%, Sharpe=-0.44, WinRate=0.00%
✓ US100_D1: Return=0.00%, DD=-0.00%, Sharpe=nan, WinRate=nan%
✓ US100_D1: Return=-0.63%, DD=-0.63%, Sharpe=-0.44, WinRate=0.00%
✓ US500_D1: Return=1.41%, DD=-0.91%, Sharpe=0.51, WinRate=85.71%
✓ US500_D1: Return=3.94%, DD=-3.29%, Sharpe=0.47, WinRate=75.68%
✓ US500_D1: Return=3.12%, DD=-0.90%, Sharpe=0.73, WinRate=78.57%
✓ US500_D1: Return=9.59%, DD=-2.85%, Sharpe=0.73, WinRate=64.86%
✓ US500_D1: Return=1.77%, DD=-2.28%, Sharpe=0.25, WinRate=42.86%
✓ US500_D1: Return=6.06%, DD=-4.34%, Sharpe=0.31, WinRate=40.54%
✓ US500_D1: Return=2.79%, DD=-2.27%, Sharpe=0.35, WinRate=35.71%
✓ US500_D1: Return=10.46%, DD=-4.73%, Sharpe=0.44, WinRate=32.43%

Backtesting complete!
Results stored in ../backtest_results.db


### Summary

In [10]:
query_pf = f"""
SELECT asset, timeframe, risk_reward, return_pct, profit_factor, win_rate_pct, num_trades, equity_final
FROM {table_name}
WHERE profit_factor IS NOT NULL and num_trades >= 1000
ORDER BY profit_factor DESC
LIMIT 20
"""
top_pf = pd.read_sql_query(query_pf, conn)

query_wr = f"""
SELECT asset, timeframe, risk_reward, return_pct, profit_factor, win_rate_pct, num_trades, equity_final
FROM {table_name}
WHERE win_rate_pct IS NOT NULL and num_trades >= 1000
ORDER BY win_rate_pct DESC
LIMIT 20
"""
top_wr = pd.read_sql_query(query_wr, conn)

print("Top 20 by Profit Factor")
print(top_pf.to_string(index=False))
print(f"Average profit factor (top 20): {top_pf['profit_factor'].mean():.4f}")
print(f"Average win rate (top 20 by profit factor): {top_pf['win_rate_pct'].mean():.2f}%")
print("\nTop 20 by Win Rate")
print(top_wr.to_string(index=False))
print(f"Average profit factor (top 20 by win rate): {top_wr['profit_factor'].mean():.4f}")
print(f"Average win rate (top 20): {top_wr['win_rate_pct'].mean():.2f}%")


Top 20 by Profit Factor
 asset timeframe  risk_reward  return_pct  profit_factor  win_rate_pct  num_trades  equity_final
 US500        H1          3.0  -23.414722       1.047935     26.357712        1381   7658.527798
AUDUSD        H1          3.0    8.918868       1.029890     27.420325        1663  10891.886825
AUDUSD        H1          3.0    6.541385       1.029890     27.420325        1663  10654.138508
AUDUSD        H1          1.0   -9.219528       1.024354     48.798077        1664   9078.047172
AUDUSD        H1          1.0  -21.354878       1.024354     48.798077        1664   7864.512163
 US500        H1          3.0  -52.450801       1.011704     26.515152        1452   4754.919902
 US500        H1          2.0  -23.828235       1.008116     32.078204        1381   7617.176504
AUDUSD        H1          2.0   -3.091803       1.007415     33.473558        1664   9690.819669
AUDUSD        H1          2.0  -13.112180       1.007415     33.473558        1664   8688.782023
 US100

In [12]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

query = f"""
SELECT max_drawdown_pct, win_rate_pct
FROM {table_name}
WHERE max_drawdown_pct IS NOT NULL AND win_rate_pct IS NOT NULL and num_trades >= 1000
"""
df_stats = pd.read_sql_query(query, conn)

mean_dd = df_stats['max_drawdown_pct'].mean()
mean_wr = df_stats['win_rate_pct'].mean()

print(f"Mean max drawdown (%): {mean_dd:.4f}")
print(f"Mean win rate (%): {mean_wr:.4f}")

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["Distribution of Max Drawdown (%)", "Distribution of Win Rate (%)"]
)

fig.add_trace(go.Histogram(x=df_stats["max_drawdown_pct"], name="Max Drawdown", nbinsx=40), row=1, col=1)
fig.add_trace(go.Histogram(x=df_stats["win_rate_pct"], name="Win Rate", nbinsx=40), row=1, col=2)

fig.update_layout(
    title="Interactive Backtest Distributions",
    bargap=0.1,
    template="plotly_white"
)
fig.show()

Mean max drawdown (%): -92.5851
Mean win rate (%): 39.6389


In [ ]:
bt.plot(open_browser=False, filename=f"{table_name}.html", resample=False, plot_pl=False, plot_volume=False)